In [ ]:
import pandas as pd

#Import ABCNEWS-DATE-TEXT.CSV
data = pd.read_csv('abcnews-date-text.csv');

data_text = data[['headline_text']]

data_text['index'] = data_text.index

documents = data_text

In [ ]:
len(documents)

255640

In [ ]:
documents[:5]

,headline_text,index
0,aba decides against community broadcasting lic...,0
1,act fire witnesses must be aware of defamation,1
2,a g calls for infrastructure protection summit,2
3,air nz staff in aust strike for pay rise,3
4,air nz strike to affect australian travellers,4


### Data Preprocessing

In [ ]:
pip install gensim

In [ ]:
import gensim
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
import numpy as np
np.random.seed(2018)

In [ ]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

#### Lemmatize example

In [ ]:
print(WordNetLemmatizer().lemmatize('went', pos='v'))

go


#### Stemmer Example

In [ ]:
stemmer = SnowballStemmer('english')
original_words = ['caresses', 'flies', 'dies', 'mules', 'denied','died', 'agreed', 'owned',
           'humbled', 'sized','meeting', 'stating', 'siezing', 'itemization','sensational',
           'traditional', 'reference', 'colonizer','plotted']
singles = [stemmer.stem(plural) for plural in original_words]
pd.DataFrame(data = {'original word': original_words, 'stemmed': singles})

,original word,stemmed
0,caresses,caress
1,flies,fli
2,dies,die
3,mules,mule
4,denied,deni
5,died,die
6,agreed,agre
7,owned,own
8,humbled,humbl
9,sized,size


In [ ]:
def lemmatize_stemming(text):
    return stemmer.stem(WordNetLemmatizer().lemmatize(text, pos='v'))

def preprocess(text):
    result = []
    for token in gensim.utils.simple_preprocess(text):
        if token not in gensim.parsing.preprocessing.STOPWORDS and len(token) > 3:
            result.append(lemmatize_stemming(token))
    return result

In [ ]:
doc_sample = documents[documents['index'] == 4310].values[0][0]

print('original document: ')
words = []
for word in doc_sample.split(' '):
    words.append(word)
print(words)
print('\n\n tokenized and lemmatized document: ')
print(preprocess(doc_sample))

original document: 
['rain', 'helps', 'dampen', 'bushfires']


 tokenized and lemmatized document: 
['rain', 'help', 'dampen', 'bushfir']


In [ ]:
# Fill any non-string values in 'headline_text' with an empty string
documents['headline_text'] = documents['headline_text'].fillna('')

In [ ]:
processed_docs = documents['headline_text'].map(preprocess)

In [ ]:
processed_docs[:10]

,headline_text
0,"[decid, communiti, broadcast, licenc]"
1,"[wit, awar, defam]"
2,"[call, infrastructur, protect, summit]"
3,"[staff, aust, strike, rise]"
4,"[strike, affect, australian, travel]"
5,"[ambiti, olsson, win, tripl, jump]"
6,"[antic, delight, record, break, barca]"
7,"[aussi, qualifi, stosur, wast, memphi, match]"
8,"[aust, address, secur, council, iraq]"
9,"[australia, lock, timet]"


### Bag of words on the dataset

In [ ]:
dictionary = gensim.corpora.Dictionary(processed_docs)

In [ ]:
count = 0
for k, v in dictionary.iteritems():
    print(k, v)
    count += 1
    if count > 10:
        break

0 broadcast
1 communiti
2 decid
3 licenc
4 awar
5 defam
6 wit
7 call
8 infrastructur
9 protect
10 summit


In [ ]:
dictionary.filter_extremes(no_below=15, no_above=0.5, keep_n=100000)

In [ ]:
bow_corpus = [dictionary.doc2bow(doc) for doc in processed_docs]
bow_corpus[4310]

[(71, 1), (107, 1), (459, 1), (3446, 1)]

In [ ]:
bow_doc_4310 = bow_corpus[4310]

for i in range(len(bow_doc_4310)):
    print("Word {} (\"{}\") appears {} time.".format(bow_doc_4310[i][0],
                                                     dictionary[bow_doc_4310[i][0]],
                                                     bow_doc_4310[i][1]))

Word 71 ("bushfir") appears 1 time.
Word 107 ("help") appears 1 time.
Word 459 ("rain") appears 1 time.
Word 3446 ("dampen") appears 1 time.


### TF-IDF

In [ ]:
from gensim import corpora, models

tfidf = models.TfidfModel(bow_corpus)

In [ ]:
corpus_tfidf = tfidf[bow_corpus]

In [ ]:
from pprint import pprint

for doc in corpus_tfidf:
    pprint(doc)
    break

[(0, np.float64(0.5910830220697492)),
 (1, np.float64(0.3941920782722793)),
 (2, np.float64(0.4874548105605509)),
 (3, np.float64(0.5075640591192908))]


### Running LDA using Bag of Words

In [ ]:
lda_model = gensim.models.LdaMulticore(bow_corpus, num_topics=10, id2word=dictionary, passes=2, workers=2)

In [ ]:
for idx, topic in lda_model.print_topics(-1):
    print('Topic: {} \nWords: {}'.format(idx, topic))

Topic: 0 
Words: 0.063*"polic" + 0.038*"charg" + 0.032*"face" + 0.029*"court" + 0.023*"investig" + 0.021*"accus" + 0.018*"murder" + 0.015*"elect" + 0.014*"woman" + 0.011*"bodi"
Topic: 1 
Words: 0.018*"power" + 0.017*"labor" + 0.015*"terror" + 0.015*"sale" + 0.013*"govt" + 0.013*"case" + 0.013*"howard" + 0.012*"farmer" + 0.012*"warn" + 0.011*"say"
Topic: 2 
Words: 0.038*"govt" + 0.031*"plan" + 0.028*"council" + 0.025*"urg" + 0.022*"water" + 0.018*"fund" + 0.014*"health" + 0.013*"seek" + 0.012*"group" + 0.011*"opposit"
Topic: 3 
Words: 0.027*"hospit" + 0.020*"talk" + 0.020*"home" + 0.018*"aust" + 0.014*"south" + 0.014*"head" + 0.014*"timor" + 0.013*"leav" + 0.013*"cost" + 0.013*"east"
Topic: 4 
Words: 0.040*"kill" + 0.027*"attack" + 0.024*"jail" + 0.019*"death" + 0.017*"bomb" + 0.016*"polic" + 0.015*"driver" + 0.013*"dead" + 0.011*"assault" + 0.011*"injur"
Topic: 5 
Words: 0.026*"rise" + 0.021*"concern" + 0.020*"price" + 0.018*"fear" + 0.017*"nation" + 0.014*"reject" + 0.014*"communiti" 

Cool! Can you distinguish different topics using the words in each topic and their corresponding weights?

### Running LDA using TF-IDF

In [ ]:
lda_model_tfidf = gensim.models.LdaMulticore(corpus_tfidf, num_topics=10, id2word=dictionary, passes=2, workers=4)

In [ ]:
for idx, topic in lda_model_tfidf.print_topics(-1):
    print('Topic: {} Word: {}'.format(idx, topic))

Topic: 0 Word: 0.008*"aussi" + 0.008*"final" + 0.007*"world" + 0.007*"lead" + 0.007*"test" + 0.006*"england" + 0.006*"tiger" + 0.006*"australia" + 0.006*"open" + 0.006*"win"
Topic: 1 Word: 0.005*"mark" + 0.005*"news" + 0.005*"cowboy" + 0.005*"produc" + 0.005*"quit" + 0.005*"thousand" + 0.004*"origin" + 0.004*"collaps" + 0.004*"human" + 0.004*"pay"
Topic: 2 Word: 0.010*"guilti" + 0.007*"plead" + 0.005*"contract" + 0.005*"socceroo" + 0.005*"retir" + 0.005*"drink" + 0.005*"govt" + 0.005*"stand" + 0.005*"sale" + 0.004*"croc"
Topic: 3 Word: 0.021*"polic" + 0.020*"kill" + 0.017*"charg" + 0.012*"crash" + 0.011*"investig" + 0.011*"court" + 0.011*"murder" + 0.010*"death" + 0.010*"attack" + 0.009*"bomb"
Topic: 4 Word: 0.011*"miss" + 0.010*"search" + 0.010*"plan" + 0.009*"council" + 0.008*"concern" + 0.007*"farm" + 0.007*"worri" + 0.007*"water" + 0.007*"polic" + 0.006*"shortag"
Topic: 5 Word: 0.009*"govt" + 0.008*"council" + 0.008*"plan" + 0.007*"blaze" + 0.007*"urg" + 0.006*"bushfir" + 0.006*"ch

### Classification of the topics

### Performance evaluation by classifying sample document using LDA Bag of Words model

In [ ]:
processed_docs[4310]

['rain', 'help', 'dampen', 'bushfir']

In [ ]:
for index, score in sorted(lda_model[bow_corpus[4310]], key=lambda tup: -1*tup[1]):
    print("\nScore: {}\t \nTopic: {}".format(score, lda_model.print_topic(index, 10)))


Score: 0.8199734687805176	 
Topic: 0.028*"help" + 0.027*"open" + 0.025*"miss" + 0.020*"school" + 0.016*"coast" + 0.015*"boost" + 0.015*"search" + 0.014*"gold" + 0.014*"doctor" + 0.012*"drought"

Score: 0.020007897168397903	 
Topic: 0.020*"law" + 0.017*"secur" + 0.017*"hous" + 0.016*"crash" + 0.012*"highway" + 0.012*"melbourn" + 0.011*"blaze" + 0.011*"prompt" + 0.011*"probe" + 0.011*"port"

Score: 0.020002875477075577	 
Topic: 0.038*"govt" + 0.031*"plan" + 0.028*"council" + 0.025*"urg" + 0.022*"water" + 0.018*"fund" + 0.014*"health" + 0.013*"seek" + 0.012*"group" + 0.011*"opposit"

Score: 0.02000225894153118	 
Topic: 0.018*"market" + 0.015*"strike" + 0.013*"england" + 0.011*"abus" + 0.010*"victori" + 0.010*"share" + 0.010*"illeg" + 0.010*"record" + 0.010*"challeng" + 0.009*"make"

Score: 0.020002255216240883	 
Topic: 0.063*"polic" + 0.038*"charg" + 0.032*"face" + 0.029*"court" + 0.023*"investig" + 0.021*"accus" + 0.018*"murder" + 0.015*"elect" + 0.014*"woman" + 0.011*"bodi"

Score: 0.0

Our test document has the highest probability to be part of the topic on the top.

### Performance evaluation by classifying sample document using LDA TF-IDF model

In [ ]:
for index, score in sorted(lda_model_tfidf[bow_corpus[4310]], key=lambda tup: -1*tup[1]):
    print("\nScore: {}\t \nTopic: {}".format(score, lda_model_tfidf.print_topic(index, 10)))


Score: 0.33355939388275146	 
Topic: 0.009*"govt" + 0.008*"council" + 0.008*"plan" + 0.007*"blaze" + 0.007*"urg" + 0.006*"bushfir" + 0.006*"chang" + 0.006*"firefight" + 0.006*"beazley" + 0.005*"dump"

Score: 0.2632575035095215	 
Topic: 0.011*"coast" + 0.008*"gold" + 0.007*"rain" + 0.007*"cyclon" + 0.007*"grower" + 0.007*"rat" + 0.006*"north" + 0.005*"west" + 0.005*"south" + 0.004*"central"

Score: 0.2631283700466156	 
Topic: 0.025*"closer" + 0.006*"rescu" + 0.005*"light" + 0.005*"come" + 0.005*"discuss" + 0.005*"heritag" + 0.004*"committe" + 0.004*"chopper" + 0.004*"step" + 0.004*"surgeri"

Score: 0.02001134492456913	 
Topic: 0.014*"fund" + 0.012*"health" + 0.009*"govt" + 0.009*"servic" + 0.008*"centr" + 0.008*"boost" + 0.007*"plan" + 0.006*"budget" + 0.006*"region" + 0.006*"urg"

Score: 0.020008394494652748	 
Topic: 0.011*"miss" + 0.010*"search" + 0.010*"plan" + 0.009*"council" + 0.008*"concern" + 0.007*"farm" + 0.007*"worri" + 0.007*"water" + 0.007*"polic" + 0.006*"shortag"

Score: 0

Our test document has the highest probability to be part of the topic on the top.

### Testing model on unseen document

In [ ]:
unseen_document = 'How a Pentagon deal became an identity crisis for Google'
bow_vector = dictionary.doc2bow(preprocess(unseen_document))

for index, score in sorted(lda_model[bow_vector], key=lambda tup: -1*tup[1]):
    print("Score: {}\t Topic: {}".format(score, lda_model.print_topic(index, 5)))

Score: 0.2763338088989258	 Topic: 0.027*"hospit" + 0.020*"talk" + 0.020*"home" + 0.018*"aust" + 0.014*"south"
Score: 0.22848668694496155	 Topic: 0.020*"law" + 0.017*"secur" + 0.017*"hous" + 0.016*"crash" + 0.012*"highway"
Score: 0.211665078997612	 Topic: 0.018*"market" + 0.015*"strike" + 0.013*"england" + 0.011*"abus" + 0.010*"victori"
Score: 0.1834423542022705	 Topic: 0.040*"kill" + 0.027*"attack" + 0.024*"jail" + 0.019*"death" + 0.017*"bomb"
Score: 0.016681469976902008	 Topic: 0.018*"power" + 0.017*"labor" + 0.015*"terror" + 0.015*"sale" + 0.013*"govt"
Score: 0.016679057851433754	 Topic: 0.038*"govt" + 0.031*"plan" + 0.028*"council" + 0.025*"urg" + 0.022*"water"
Score: 0.016678860411047935	 Topic: 0.018*"win" + 0.016*"closer" + 0.014*"final" + 0.014*"aussi" + 0.014*"world"
Score: 0.016678379848599434	 Topic: 0.028*"help" + 0.027*"open" + 0.025*"miss" + 0.020*"school" + 0.016*"coast"
Score: 0.01667713187634945	 Topic: 0.063*"polic" + 0.038*"charg" + 0.032*"face" + 0.029*"court" + 0.02